### Droughts, reelection, and insurance

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import geopandas as gpd

In [2]:
df_catnat = pd.read_csv('../data/gaspar/catnat_gaspar.csv', sep=';')

df_catnat['dat_deb'] = pd.to_datetime(df_catnat['dat_deb'], format='%Y-%m-%d')
df_catnat['dat_pub_jo'] = pd.to_datetime(df_catnat['dat_pub_jo'], format='%Y-%m-%d')
df_catnat['duree'] = (df_catnat['dat_pub_jo'] - df_catnat['dat_deb']).dt.days

df_catnat[['duree', 'lib_risque_jo']].groupby('lib_risque_jo').agg(['count', 'mean', 'std']).sort_values(('duree', 'count'), ascending=False).head(10)

duree              \
                                                     count        mean   
lib_risque_jo                                                            
Inondations et/ou Coulées de Boue                   145433   70.705369   
Sécheresse                                           46365  826.412984   
Mouvement de Terrain                                 32298   37.989287   
Tempête                                              16187   20.690678   
Chocs Mécaniques liés à l'action des Vagues           6773   25.462720   
Glissement de Terrain                                 3903  107.143992   
Poids de la Neige                                     2758  113.748731   
Mouvements de terrain différentiels consécutifs...    1543  588.787427   
Grêle                                                 1491   56.592220   
Inondations Remontée Nappe                            1430  319.219580   

                                                                
                                                           std  
lib_risque_jo                                                   
Inondations et/ou Coulées de Boue                    87.550285  
Sécheresse                                          715.050420  
Mouvement de Terrain                                130.691360  
Tempête                                              18.063089  
Chocs Mécaniques liés à l'action des Vagues          59.067270  
Glissement de Terrain                               138.837226  
Poids de la Neige                                    95.692776  
Mouvements de terrain différentiels consécutifs...  114.039780  
Grêle                                                20.345375  
Inondations Remontée Nappe                          248.694384

In [72]:
df_droughts = df_catnat[df_catnat['lib_risque_jo'] == 'Sécheresse']

In [220]:
df_spei = pd.read_csv('C:/Users/colin/Downloads/spei.csv', sep=',')
df_anomaly_temp = pd.read_csv('C:/Users/colin/Downloads/temperature_anomaly.csv', sep=',')
df_spei['lat'] = df_spei['ID'].str.split('_').str[0].astype(float)
df_spei['lon'] = df_spei['ID'].str.split('_').str[1].astype(float)
df_anomaly_temp['lat'] = df_anomaly_temp['ID'].str.split('_').str[0].astype(float)
df_anomaly_temp['lon'] = df_anomaly_temp['ID'].str.split('_').str[1].astype(float)

In [221]:
df_meta = pd.read_csv('C:/Users/colin/Downloads/coordonnees_grille_safran_lambert-2-etendu.csv', sep=';')
df_spei = df_spei.dropna()
df_meta['LAT_DG'] = df_meta['LAT_DG'].str.replace(',', '.')
df_meta['LAT_DG'] = df_meta['LAT_DG'].astype(float)
df_meta['LON_DG'] = df_meta['LON_DG'].str.replace(',', '.')
df_meta['LON_DG'] = df_meta['LON_DG'].astype(float)
df_meta['LAMBX (hm)'] = df_meta['LAMBX (hm)'].astype(float)
df_meta['LAMBY (hm)'] = df_meta['LAMBY (hm)'].astype(float)
df_spei = df_spei.merge(df_meta, left_on=['lat', 'lon'], right_on=['LAMBX (hm)', 'LAMBY (hm)'], how='left')
df_anomaly_temp = df_anomaly_temp.merge(df_meta, left_on=['lat', 'lon'], right_on=['LAMBX (hm)', 'LAMBY (hm)'], how='left')
df_meta['ID'] = df_meta['LAMBX (hm)'].astype(int).astype(str) + '_' + df_meta['LAMBY (hm)'].astype(int).astype(str)

In [229]:
df_spei['time'] = pd.to_datetime(df_spei['time'], format='%Y-%m-%d %H:%M:%S')

In [7]:
lau_2023 = gpd.read_file('../data/LAU_2023_EU/lau_2023_final.shp')
lau_2023 = lau_2023[lau_2023.CNTR_CODE == 'FR']
lau_2023 = lau_2023.to_crs(epsg=4326)

In [39]:
#find the closest ID in df_meta of each centroid of the city
lau_2023['x'] = lau_2023.geometry.centroid.x
lau_2023['y'] = lau_2023.geometry.centroid.y

def find_closest_id(row, df_meta):
    # Calculate the distance between the row's coordinates and all coordinates in df_meta
    distances = np.sqrt((df_meta['LAT_DG'] - row['y'])**2 + (df_meta['LON_DG'] - row['x'])**2)
    # Find the index of the closest point
    closest_index = distances.idxmin()
    # Return the ID of the closest point
    return df_meta.loc[closest_index, 'ID']

lau_2023['closest_id'] = lau_2023.apply(find_closest_id, axis=1, df_meta=df_meta)

C:\Users\colin\AppData\Local\Temp\ipykernel_50752\696722185.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  lau_2023['x'] = lau_2023.geometry.centroid.x
C:\Users\colin\AppData\Local\Temp\ipykernel_50752\696722185.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  lau_2023['y'] = lau_2023.geometry.centroid.y


In [9]:
df_droughts

,cod_nat_catnat,cod_commune,lib_commune,num_risque_jo,lib_risque_jo,dat_deb,dat_fin,dat_pub_arrete,dat_pub_jo,dat_maj,duree
5490,INTE0000771A,04001,Aiglun,59.0,Sécheresse,1998-01-01,1999-09-30,2000-12-27,2000-12-29,2022-05-24,1093
5491,INTE0000771A,04047,Champtercier,59.0,Sécheresse,1997-04-01,1998-06-30,2000-12-27,2000-12-29,2022-05-24,1368
5492,INTE0000771A,04128,Montfuron,59.0,Sécheresse,1997-04-01,1999-07-31,2000-12-27,2000-12-29,2022-05-24,1368
5493,INTE0000771A,09061,Les Bordes-sur-Arize,59.0,Sécheresse,1989-05-01,1990-12-31,2000-12-27,2000-12-29,2022-05-24,4260
5494,INTE0000771A,09307,Taurignan-Castet,59.0,Sécheresse,1989-05-01,1990-12-31,2000-12-27,2000-12-29,2022-05-24,4260
...,...,...,...,...,...,...,...,...,...,...,...
259848,INTE2430295A,31478,Saint-Félix-Lauragais,NaN,Sécheresse,2023-03-31,2023-06-29,2024-11-18,2024-12-02,2024-12-04,612
259855,INTE2430295A,11281,Pexiora,NaN,Sécheresse,2023-09-30,2023-12-30,2024-11-18,2024-12-02,2024-12-04,429
259858,INTE2430295A,66069,Espira-de-l'Agly,NaN,Sécheresse,2022-12-31,2023-12-30,2024-11-18,2024-12-02,2024-12-04,702
259860,INTE2430295A,26153,Laborel,NaN,Sécheresse,2023-03-31,2023-06-29,2024-11-18,2024-12-02,2024-12-04,612


In [17]:
df_spei

,ID,year,month,time,def_pr,spei,lat,lon,LAMBX (hm),LAMBY (hm),LAT_DG,LON_DG
0,1000_23290,1970,1,1970-01-16 00:00:00,2.699792,1.142342,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
1,1000_23290,1970,2,1970-02-14 12:00:00,2.380476,1.007828,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
2,1000_23290,1970,3,1970-03-16 00:00:00,0.309462,0.076555,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
3,1000_23290,1970,4,1970-04-15 12:00:00,-0.294667,-0.238048,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
4,1000_23290,1970,5,1970-05-16 00:00:00,-1.796452,-1.019978,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
...,...,...,...,...,...,...,...,...,...,...,...,...
6568283,9960_25050,2024,12,2024-12-16 00:00:00,2.274086,1.019817,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402
6568284,9960_25050,2025,1,2025-01-16 00:00:00,1.936022,0.839519,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402
6568285,9960_25050,2025,2,2025-02-14 12:00:00,0.943690,0.310207,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402
6568286,9960_25050,2025,3,2025-03-16 00:00:00,-1.147043,-0.925408,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402


In [ ]:
# #for every row in df_droughts extract the mean SPEI from df_spei for the corresponding closest_id and date

# def extract_spei(row, df_spei):
#     year_start = row['dat_deb'].year -1
#     month_start = row['dat_deb'].month
#     year_end = row['dat_pub_jo'].year
#     month_end = row['dat_pub_jo'].month

#     #join based on lau_2023
#     closest_id = lau_2023[lau_2023['LAU_ID']==row['cod_commune']]['closest_id'].values[0]
#     df_spei_filtered = df_spei[df_spei['ID'] == closest_id]
#     df_spei_filtered = df_spei_filtered[(df_spei_filtered['year'] >= year_start) & (df_spei_filtered['year'] <= year_end)]
#     df_spei_filtered = df_spei_filtered[((df_spei_filtered['year'] != year_start)) | (df_spei_filtered['month'] < month_start)]
#     df_spei_filtered = df_spei_filtered[((df_spei_filtered['year'] != year_end)) | (df_spei_filtered['month'] > month_end)]
#     spei = df_spei_filtered['spei'].mean()
#     return spei


In [ ]:
from tqdm import tqdm
df_spei

,ID,year,month,time,def_pr,spei,lat,lon,LAMBX (hm),LAMBY (hm),LAT_DG,LON_DG
0,1000_23290,1970,1,1970-01-16 00:00:00,2.699792,1.142342,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
1,1000_23290,1970,2,1970-02-14 12:00:00,2.380476,1.007828,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
2,1000_23290,1970,3,1970-03-16 00:00:00,0.309462,0.076555,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
3,1000_23290,1970,4,1970-04-15 12:00:00,-0.294667,-0.238048,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
4,1000_23290,1970,5,1970-05-16 00:00:00,-1.796452,-1.019978,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
...,...,...,...,...,...,...,...,...,...,...,...,...
6568283,9960_25050,2024,12,2024-12-16 00:00:00,2.274086,1.019817,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402
6568284,9960_25050,2025,1,2025-01-16 00:00:00,1.936022,0.839519,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402
6568285,9960_25050,2025,2,2025-02-14 12:00:00,0.943690,0.310207,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402
6568286,9960_25050,2025,3,2025-03-16 00:00:00,-1.147043,-0.925408,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402


In [63]:
from tqdm import tqdm

def extract_spei_optimized(df_droughts, df_spei, lau_2023):
    # Merge closest_id into df_droughts
    df_droughts = df_droughts.merge(lau_2023[['LAU_ID', 'closest_id']],
                                     left_on='cod_commune', right_on='LAU_ID',
                                     how='left')
    print(df_droughts.shape)
    # Preprocess SPEI
    #df_spei = preprocess_spei(df_spei)
    df_spei['time'] = pd.to_datetime(df_spei['time'], format='%Y-%m-%d %H:%M:%S')
    def get_spei(row):
        start_date = pd.to_datetime(row['dat_deb']) - pd.DateOffset(years=1)
        end_date = pd.to_datetime(row['dat_fin'])
        mask = (
            (df_spei['ID'] == row['closest_id']) &
            (df_spei['time'] >= start_date) &
            (df_spei['time'] <= end_date)
        )
        return df_spei.loc[mask, 'spei'].mean(), df_spei.loc[mask, 'spei'].min(), df_spei.loc[mask, 'spei'].max()
    tqdm.pandas()
    df_droughts[['mean_spei', 'min_spei', 'max_spei']] = df_droughts.progress_apply(get_spei, axis=1, result_type='expand')
    
    return df_droughts


In [52]:
df_droughts

,cod_nat_catnat,cod_commune,lib_commune,num_risque_jo,lib_risque_jo,dat_deb,dat_fin,dat_pub_arrete,dat_pub_jo,dat_maj,duree
5490,INTE0000771A,04001,Aiglun,59.0,Sécheresse,1998-01-01,1999-09-30,2000-12-27,2000-12-29,2022-05-24,1093
5491,INTE0000771A,04047,Champtercier,59.0,Sécheresse,1997-04-01,1998-06-30,2000-12-27,2000-12-29,2022-05-24,1368
5492,INTE0000771A,04128,Montfuron,59.0,Sécheresse,1997-04-01,1999-07-31,2000-12-27,2000-12-29,2022-05-24,1368
5493,INTE0000771A,09061,Les Bordes-sur-Arize,59.0,Sécheresse,1989-05-01,1990-12-31,2000-12-27,2000-12-29,2022-05-24,4260
5494,INTE0000771A,09307,Taurignan-Castet,59.0,Sécheresse,1989-05-01,1990-12-31,2000-12-27,2000-12-29,2022-05-24,4260
...,...,...,...,...,...,...,...,...,...,...,...
259848,INTE2430295A,31478,Saint-Félix-Lauragais,NaN,Sécheresse,2023-03-31,2023-06-29,2024-11-18,2024-12-02,2024-12-04,612
259855,INTE2430295A,11281,Pexiora,NaN,Sécheresse,2023-09-30,2023-12-30,2024-11-18,2024-12-02,2024-12-04,429
259858,INTE2430295A,66069,Espira-de-l'Agly,NaN,Sécheresse,2022-12-31,2023-12-30,2024-11-18,2024-12-02,2024-12-04,702
259860,INTE2430295A,26153,Laborel,NaN,Sécheresse,2023-03-31,2023-06-29,2024-11-18,2024-12-02,2024-12-04,612


In [222]:
df_reelection = pd.read_csv('../data/reelection_muni.csv')


In [223]:
list_date_election = ['28/06/2020', '30/03/2014', '16/03/2008']
list_date_election = [pd.to_datetime(date, format='%d/%m/%Y') for date in list_date_election]

#show the number of days between the date of the election and the next election
for i in range(len(list_date_election)-1):
    print((list_date_election[i+1] - list_date_election[i]).days)

# df_droughts = df_droughts[df_droughts['lib_risque_jo'] == 'Inondations et/ou Coulées de Boue']
# df_droughts = df_droughts[df_droughts['duree'] < 600]
df_final = pd.DataFrame()
df_droughts = df_droughts.drop_duplicates()

for i in range(len(list_date_election) - 1):
    df_temp = df_droughts[(df_droughts['dat_pub_jo'] < list_date_election[i]) & (df_droughts['dat_pub_jo'] > list_date_election[i + 1]) & (df_droughts['dat_deb'] > list_date_election[i + 1])].copy()
    one_year = pd.Timedelta(days=365)
    df_temp.loc[:, 'mandat'] = 1
    df_temp.loc[:, 'mandat_1'] = ((list_date_election[i] - df_temp['dat_pub_jo']) < (list_date_election[i] - list_date_election[i + 1] - one_year)) 
    df_temp.loc[:, 'mandat_2'] = ((list_date_election[i] - df_temp['dat_pub_jo']) < (list_date_election[i] - list_date_election[i + 1] - 2 * one_year))
    df_temp.loc[:, 'mandat_3'] = ((list_date_election[i] - df_temp['dat_pub_jo']) < (list_date_election[i] - list_date_election[i + 1] - 3 * one_year))
    df_temp.loc[:, 'mandat_4'] = ((list_date_election[i] - df_temp['dat_pub_jo']) < (list_date_election[i] - list_date_election[i + 1] - 4 * one_year))
    df_temp = df_temp[['cod_commune', 'lib_commune', 'mandat', 'mandat_1', 'mandat_2', 'mandat_3', 'mandat_4']].groupby(['cod_commune', 'lib_commune']).sum().reset_index()
    df_temp.loc[:, 'election'] = list_date_election[i]
    df_final = pd.concat([df_final, df_temp], axis=0)

df_final.to_csv('../data/catnat_droughts_gaspar_mandat_muni.csv', sep=';', index=False)

-2282
-2205


In [224]:
df_final = df_final[['cod_commune', 'lib_commune', 'mandat_4', 'election']]
df_final = df_final.pivot(index='cod_commune', columns='election', values='mandat_4')
df_final = df_final.reset_index()
df_final.columns = ['cod_commune', 'catnat_2014', 'catnat_2020']

In [280]:
df_spei_election = pd.DataFrame()   
for i, election_date in enumerate(list_date_election):
    start = election_date - pd.DateOffset(month=3)
    end = election_date
    df_spei_temp = df_spei[(df_spei['time'] >= start) & (df_spei['time'] <= end)]
    df_spei_temp = df_spei_temp[['ID','spei']].groupby(['ID']).agg(['mean', 'min', 'max']).reset_index()
    # df_spei_temp = df_spei_temp.rename(columns={'mean': 'mean_spei', 'min': 'min_spei', 'max': 'max_spei'})
    df_spei_temp['election'] = election_date    
    
    df_spei_election = pd.concat([df_spei_election, df_spei_temp], axis=0)
df_spei_election.reset_index(drop=True, inplace=True)
#concatenate the multiindex columns into a single column
df_spei_election.columns = ['_'.join(col).strip() for col in df_spei_election.columns.values]
#remove _ at the end of the column names
df_spei_election.columns = df_spei_election.columns.str.rstrip('_')

df_spei_election['droughts'] = (df_spei_election.spei_mean <-1) & (df_spei_election.spei_min<(-1*df_spei_election.spei_max)) & (df_spei_election.spei_min <-1)

In [272]:
df_spei_election.droughts.mean()

0.08289526890416499

In [244]:
df_spei_election.election.unique(), list_date_election

(<DatetimeArray>
 ['2020-06-28 00:00:00', '2008-03-16 00:00:00']
 Length: 2, dtype: datetime64[ns],
 [Timestamp('2020-06-28 00:00:00'),
  Timestamp('2014-03-30 00:00:00'),
  Timestamp('2008-03-16 00:00:00')])

In [281]:
#pivot to keep one droughts column for each election
df_spei_election = df_spei_election.pivot(index='ID', columns='election', values='droughts').reset_index()
df_spei_election.columns = ['ID'] + ['spei_' + str(col.year) for col in df_spei_election.columns[1:]]
try:
    lau_droughts_spei = lau_2023.merge(df_spei_election[['ID', 'spei_2008', 'spei_2014', 'spei_2020']],
                                   left_on='closest_id', right_on='ID', how='left')
except:
    lau_droughts_spei = lau_2023.merge(df_spei_election[['ID', 'spei_2008', 'spei_2020']],
                                   left_on='closest_id', right_on='ID', how='left')

In [282]:
df_anomaly = pd.DataFrame()
df_anomaly_temp['time'] = pd.to_datetime(df_anomaly_temp['year'].astype(str) + '-' + df_anomaly_temp['month'].astype(str) + '-01', format='%Y-%m-%d')

for i, election_date in enumerate(list_date_election):
    start = election_date - pd.DateOffset(years=1)
    end = election_date
    df_temp = df_anomaly_temp[(df_anomaly_temp['time'] >= start) & (df_anomaly_temp['time'] <= end)]
    df_temp = df_temp[df_temp.month.isin([6, 7, 8])]
    df_temp = df_temp[['ID','positive_anomaly']].groupby(['ID']).agg(['mean', 'max']).reset_index()
    # df_spei_temp = df_spei_temp.rename(columns={'mean': 'mean_spei', 'min': 'min_spei', 'max': 'max_spei'})
    df_temp['election'] = election_date    
    
    df_anomaly = pd.concat([df_anomaly, df_temp], axis=0)

df_anomaly.reset_index(drop=True, inplace=True)
#concatenate the multiindex columns into a single column
df_anomaly.columns = ['_'.join(col).strip() for col in df_anomaly.columns.values]
#remove _ at the end of the column names
df_anomaly.columns = df_anomaly.columns.str.rstrip('_')
df_anomaly

,ID,positive_anomaly_mean,positive_anomaly_max,election
0,1000_23290,5.333333,10,2020-06-28
1,1000_23370,5.333333,10,2020-06-28
2,1000_23450,5.333333,10,2020-06-28
3,1000_23530,4.666667,8,2020-06-28
4,1000_23610,5.333333,10,2020-06-28
...,...,...,...,...
29671,9960_24730,3.333333,6,2008-03-16
29672,9960_24810,3.666667,7,2008-03-16
29673,9960_24890,3.333333,6,2008-03-16
29674,9960_24970,3.333333,6,2008-03-16


In [262]:
df_anomaly['anomaly'] = (df_anomaly.positive_anomaly_max > 10) 

In [263]:
df_anomaly = df_anomaly.pivot(index='ID', columns='election', values='anomaly').reset_index()
df_anomaly.columns = ['ID'] + ['anomaly_' + str(col.year) for col in df_anomaly.columns[1:]]
lau_temp_anomaly = lau_2023.merge(df_anomaly[['ID', 'anomaly_2008', 'anomaly_2014', 'anomaly_2020']],
                                      left_on='closest_id', right_on='ID', how='left')

In [283]:
try:
    df_lau = lau_droughts_spei[['LAU_ID', 'spei_2020', 'spei_2014', 'spei_2008']].copy()
except:
    df_lau = lau_droughts_spei[['LAU_ID', 'spei_2020', 'spei_2008']].copy()
df_lau = df_lau.merge(df_final[['cod_commune', 'catnat_2014', 'catnat_2020']], left_on='LAU_ID', right_on='cod_commune', how='left')
df_lau = df_lau.merge(df_reelection[['GEO_INSEE', 'reelected_2020', 'reelected_2014']], left_on='LAU_ID', right_on='GEO_INSEE', how='left')
df_lau = df_lau.merge(lau_temp_anomaly[['LAU_ID', 'anomaly_2020', 'anomaly_2014', 'anomaly_2008']], left_on='LAU_ID', right_on='LAU_ID', how='left')

In [284]:
df_lau.fillna(0, inplace=True)
df_lau.drop(columns=['cod_commune', 'GEO_INSEE'], inplace=True)

In [286]:
#find the number of LAU_ID with spei_2020 ==

print(df_lau[(df_lau['reelected_2020'] == 1) & (df_lau['catnat_2020'] == 1)].shape[0]/df_lau[(df_lau['catnat_2020'] == 1)].shape[0])
print('for nb counties : ', df_lau[(df_lau['catnat_2020'] == 1)].shape[0])
print(df_lau[(df_lau['reelected_2020'] == 1)].shape[0]/df_lau.shape[0])
print(df_lau[(df_lau['reelected_2020'] == 1) & (df_lau['catnat_2020'] == 0) & (df_lau['spei_2020'] == 1)].shape[0]/df_lau[(df_lau['catnat_2020'] == 0) & (df_lau['spei_2020'] == 1)].shape[0])
print('for nb counties : ', df_lau[(df_lau['catnat_2020'] == 0) & (df_lau['spei_2020'] == 1)].shape[0])
print(df_lau[(df_lau['reelected_2020'] == 1) & (df_lau['catnat_2020'] == 0) & (df_lau['spei_2020'] == 1) & (df_lau['anomaly_2020'] == 1)].shape[0]/df_lau[(df_lau['catnat_2020'] == 0) & (df_lau['spei_2020'] == 1) & (df_lau['anomaly_2020'] == 1)].shape[0])
print('for nb counties : ', df_lau[(df_lau['catnat_2020'] == 0) & (df_lau['spei_2020'] == 1) & (df_lau['anomaly_2020'] == 1)].shape[0])
#same for 2014
# print(df_lau[(df_lau['reelected_2014'] == 1) & (df_lau['catnat_2014'] == 1)].shape[0]/df_lau[(df_lau['catnat_2014'] == 1)].shape[0])
# print('for nb counties : ', df_lau[(df_lau['catnat_2014'] == 1)].shape[0])
# print(df_lau[(df_lau['reelected_2014'] == 1) & (df_lau['catnat_2014'] == 0) & (df_lau['spei_2014'] == 1)].shape[0]/df_lau[(df_lau['catnat_2014'] == 0) & (df_lau['spei_2014'] == 1)].shape[0])
# print('for nb counties : ', df_lau[(df_lau['catnat_2014'] == 0) & (df_lau['spei_2014'] == 1)].shape[0])
# print(df_lau[(df_lau['reelected_2014'] == 1) & (df_lau['catnat_2014'] == 0) & (df_lau['spei_2014'] == 1) & (df_lau['anomaly_2014'] == 1)].shape[0]/df_lau[(df_lau['catnat_2014'] == 0) & (df_lau['spei_2014'] == 1) & (df_lau['anomaly_2014'] == 1)].shape[0])
# print('for nb counties : ', df_lau[(df_lau['catnat_2014'] == 0) & (df_lau['spei_2014'] == 1) & (df_lau['anomaly_2014'] == 1)].shape[0])


0.24343163538873994
for nb counties :  5595
0.19565030762626986
0.1864406779661017
for nb counties :  9971
0.20572916666666666
for nb counties :  384
